# 95662 - Introduction to Machine Learning
##### By: Brenno, Camilla, Kok Soon

## 0. Config and Imports

In [1]:
import os
from pathlib import Path
import json
import pandas as pd
import random
import math
import copy
import numpy as np

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"CUDA Available: {torch.cuda.is_available()}")

CUDA Available: True


### A. Local Setup

In [3]:
DATA_DIR = Path('./data')# Path to dataset

### B. Colab Setup

In [4]:
# !pip -q install gdown

# DATA_DIR = Path("/content/ML_Exam_Project")
# DATA_DIR.mkdir(exist_ok=True)

# FOLDER_URL = "https://drive.google.com/drive/folders/1f8t9t6jLE6vFpnu7A-xQuhieDlhk48Iw?usp=share_link"

# !gdown --folder "$FOLDER_URL" -O "$DATA_DIR"

## 1. Data

### 1.1 File Sanity Check

In [5]:
expected_files = [
    "train.jsonl",
    "validation.jsonl",
    "test_id.jsonl",
    "test_ood.jsonl",
    "test_long.jsonl",
]

for filename in expected_files:
    path = DATA_DIR / filename
    assert path.exists(), f"Missing file: {path}"

print("All dataset files found:")
for filename in expected_files:
    print(" -", DATA_DIR / filename)

All dataset files found:
 - data\train.jsonl
 - data\validation.jsonl
 - data\test_id.jsonl
 - data\test_ood.jsonl
 - data\test_long.jsonl


### 1.2 Data Loading

In [6]:
# Function to load jsonl into pandas df object
def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return pd.DataFrame(records)

In [7]:
train_df = load_jsonl(DATA_DIR / "train.jsonl")
validation_df = load_jsonl(DATA_DIR / "validation.jsonl")
test_id_df = load_jsonl(DATA_DIR / "test_id.jsonl")
test_ood_df = load_jsonl(DATA_DIR / "test_ood.jsonl")
test_long_df = load_jsonl(DATA_DIR / "test_long.jsonl")

### 1.3 Understanding Data

#### 1.3.1 Data shapes

In [8]:
print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test ID:", test_id_df.shape)
print("Test OOD:", test_ood_df.shape)
print("Test long:", test_long_df.shape)

Train: (12000, 6)
Validation: (2000, 6)
Test ID: (2000, 6)
Test OOD: (2000, 6)
Test long: (1500, 6)


#### 1.3.2 Data Example

In [9]:
train_df.head()

,id,expression,value,length,operator_count,depth
0,train-00000,4+4+7+3+8-1,25,11,5,5
1,train-00001,5-(6-5)+8+8,20,11,4,5
2,train-00002,5-3+1,3,5,2,3
3,train-00003,1-(1+5+8+7+1),-21,13,5,6
4,train-00004,7+1-(6+5+9),-12,11,4,4


#### 1.3.3 Test for White Space

It is important to test for white spaces to know if it is required to strip them away because whitespaces such as (1 + 1 3) is a wrong mathematical expression. Furthermore, our plan to use a binary tree to linearize mathematical expression may run into issues if we use whitespace to separate terms

In [10]:
def check_for_whitespace(*data_df):
    for i, df in enumerate(data_df, start=1):
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"Argument {i} is not a DataFrame")

    combined_df = pd.concat(data_df, ignore_index=True)

    return combined_df["expression"].astype(str).str.contains(r"\s").any()

In [11]:
print("White Space Exist") if check_for_whitespace(train_df,validation_df,test_id_df,test_ood_df,test_long_df) else print("No Whitespace found")

No Whitespace found


#### 1.3.4 Test for Unary Operators

Unary operators like -5 have to be considered when constructing expression trees, if unary operators exist, we have to preprocess our data in a way where the sign and the number are interpreted as one (i.e. "-5" instead of "- 5")

In [12]:
def check_for_unary_operators(*data_df):
    for i, df in enumerate(data_df, start=1):
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"Argument {i} is not a DataFrame")

    combined_df = pd.concat(data_df, ignore_index=True)

    def has_unary(expr):
        for i, ch in enumerate(expr):
            if ch in "+-":
                if i == 0:
                    return True  # starts with + or -
                prev = expr[i - 1]
                if prev in "(-+":
                    return True
        return False

    return combined_df["expression"].apply(has_unary).any()

In [13]:
print("Unary Operator Exist") if check_for_whitespace(train_df,validation_df,test_id_df,test_ood_df,test_long_df) else print("No Unary Operator found")

No Unary Operator found


#### 1.3.5 Data Preprocessing and Transformation

In this step, we aim to represent our input data in a different form in an attempt to help our models learn better

##### 1.3.5.1 ExpressionTree, Prefix, Postfix

In [14]:
# Node Structure (Intermediary structure required to build trees)
class Node:
    def __init__(self, value, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right

In [15]:
# ExpressionTree Class (class used to represent mathematical expression)
class ExpressionTree:
    def __init__(self, root_node):
        self.root_node = root_node

    def get_root_node(self):
        return self.root_node

    def to_postfix(self):
        def dfs(node):
            if node is None:
                return []
            return dfs(node.left) + dfs(node.right) + [str(node.value)]

        return " ".join(dfs(self.root_node))

    def to_prefix(self):
        def dfs(node):
            if node is None:
                return []
            return [str(node.value)] + dfs(node.left) + dfs(node.right)

        return " ".join(dfs(self.root_node))

In [16]:
def reduce(operator_stack, operand_stack):
    op = operator_stack.pop()

    right = operand_stack.pop()
    left = operand_stack.pop()

    operand_stack.append(Node(op, left, right))


def build_tree_with_expression(expression):
    subtree_stack = []
    operator_stack = []

    for ch in expression:

        if ch.isnumeric():
            subtree_stack.append(Node(ch))

        elif ch == '(':
            operator_stack.append(ch)

        elif ch == ')':
            while operator_stack[-1] != '(':
                reduce(operator_stack, subtree_stack)

            operator_stack.pop()  # remove '('

        elif ch in '+-':

            while (
                operator_stack
                and operator_stack[-1] != '('
            ):
                reduce(operator_stack, subtree_stack)

            operator_stack.append(ch)

    while operator_stack:
        reduce(operator_stack, subtree_stack)

    root = subtree_stack.pop()
    return ExpressionTree(root)

expression = "5-(6-5)+8+8"
build_tree_with_expression(expression).to_postfix()

'5 6 5 - - 8 + 8 +'

##### 1.3.5.2 Target Value Normalization
Normalizing the target variable (output) can help improve model performance and stability during training, especially with regression tasks. By scaling the target values, the model can learn more efficiently. We will normalize `y_train` and `y_val` using the mean and standard deviation calculated from `y_train`.

In [17]:
# Calculate mean and standard deviation from training targets
y_mean = train_df["value"].mean()
y_std = train_df["value"].std()

print(f"Training target mean: {y_mean:.2f}")
print(f"Training target standard deviation: {y_std:.2f}")

Training target mean: 4.86
Training target standard deviation: 10.59


In [18]:
def normalize_target_value(value,mean,std):
    return (value - mean) / std

#### 1.3.6 Applying Transformations

In this step, we apply the previously discussed preprocessing steps

In [19]:
def apply_transformation(df):
    df["normalized_value"] = df["value"].apply(lambda y: normalize_target_value(y, y_mean, y_std))
    df["prefix"] = df["expression"].apply(lambda x: build_tree_with_expression(x).to_prefix())
    df["postfix"] = df["expression"].apply(lambda x: build_tree_with_expression(x).to_postfix())

    return df

In [20]:
train_df = apply_transformation(train_df)
validation_df = apply_transformation(validation_df)
test_id_df = apply_transformation(test_id_df)
test_ood_df = apply_transformation(test_ood_df)
test_long_df = apply_transformation(test_long_df)

train_df.head()

,id,expression,value,length,operator_count,depth,normalized_value,prefix,postfix
0,train-00000,4+4+7+3+8-1,25,11,5,5,1.901590,- + + + + 4 4 7 3 8 1,4 4 + 7 + 3 + 8 + 1 -
1,train-00001,5-(6-5)+8+8,20,11,4,5,1.429479,+ + - 5 - 6 5 8 8,5 6 5 - - 8 + 8 +
2,train-00002,5-3+1,3,5,2,3,-0.175696,+ - 5 3 1,5 3 - 1 +
3,train-00003,1-(1+5+8+7+1),-21,13,5,6,-2.441825,- 1 + + + + 1 5 8 7 1,1 1 5 + 8 + 7 + 1 + -
4,train-00004,7+1-(6+5+9),-12,11,4,4,-1.592027,- + 7 1 + + 6 5 9,7 1 + 6 5 + 9 + -


#### 1.3.7 Summary of Data

Here, we create a function to give a summary of our data so we can compare the different data splits

In [21]:
# Function to return summary of data
# It is possible to calculate the summary of multiple data split together by passing them in together
def summary(*data_df):
    for i, df in enumerate(data_df, start=1):
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"Argument {i} is not a DataFrame")

    combined_df = pd.concat(data_df, ignore_index=True)

    result = pd.DataFrame({
        "mean": combined_df.mean(numeric_only=True),
        "min": combined_df.min(numeric_only=True),
        "max": combined_df.max(numeric_only=True),
    })

    # Expecting comparison of data split with different size, so we have to normalise for more meaningful comparison
    result.loc["value", "positive_ratio"] = (combined_df["value"] > 0).mean()
    result.loc["value", "negative_ratio"] = (combined_df["value"] < 0).mean()
    result.loc["value", "zero_ratio"] = (combined_df["value"] == 0).mean()

    mode = combined_df.mode(numeric_only=True).iloc[0]
    result["mode"] = mode

    return result

In [22]:
summary(train_df)

,mean,min,max,positive_ratio,negative_ratio,zero_ratio,mode
value,4.860750e+00,-36.000000,46.000000,0.65725,0.306167,0.036583,2.000000
length,9.607000e+00,5.000000,19.000000,NaN,NaN,NaN,7.000000
operator_count,3.489917e+00,2.000000,5.000000,NaN,NaN,NaN,4.000000
depth,3.689917e+00,3.000000,6.000000,NaN,NaN,NaN,4.000000
normalized_value,-3.256654e-18,-3.858156,3.884453,NaN,NaN,NaN,-0.270118


In [23]:
summary(validation_df)

,mean,min,max,positive_ratio,negative_ratio,zero_ratio,mode
value,4.585000,-32.000000,47.000000,0.64,0.323,0.037,4.000000
length,9.604000,5.000000,19.000000,NaN,NaN,NaN,7.000000
operator_count,3.497000,2.000000,5.000000,NaN,NaN,NaN,3.000000
depth,3.679000,3.000000,6.000000,NaN,NaN,NaN,3.000000
normalized_value,-0.026037,-3.480468,3.978875,NaN,NaN,NaN,-0.081274


In [24]:
summary(test_id_df)

,mean,min,max,positive_ratio,negative_ratio,zero_ratio,mode
value,5.232500,-29.000000,40.000000,0.668,0.296,0.036,8.000000
length,9.609000,5.000000,19.000000,NaN,NaN,NaN,7.000000
operator_count,3.487000,2.000000,5.000000,NaN,NaN,NaN,2.000000
depth,3.722000,3.000000,6.000000,NaN,NaN,NaN,4.000000
normalized_value,0.035101,-3.197202,3.317921,NaN,NaN,NaN,0.296414


In [25]:
summary(test_ood_df)

,mean,min,max,positive_ratio,negative_ratio,zero_ratio,mode
value,4.855500,-38.000000,44.000000,0.638,0.332,0.03,5.000000
length,12.484000,7.000000,23.000000,NaN,NaN,NaN,9.000000
operator_count,4.515500,3.000000,6.000000,NaN,NaN,NaN,5.000000
depth,4.434500,3.000000,7.000000,NaN,NaN,NaN,4.000000
normalized_value,-0.000496,-4.047001,3.695609,NaN,NaN,NaN,0.013148


In [26]:
summary(test_long_df)

,mean,min,max,positive_ratio,negative_ratio,zero_ratio,mode
value,4.024667,-42.000000,51.000000,0.609333,0.362,0.028667,7.000000
length,17.744000,15.000000,27.000000,NaN,NaN,NaN,15.000000
operator_count,6.214667,5.000000,7.000000,NaN,NaN,NaN,7.000000
depth,5.500667,4.000000,8.000000,NaN,NaN,NaN,4.000000
normalized_value,-0.078945,-4.424689,4.356563,NaN,NaN,NaN,0.201992


## 2. CNN Regression

### 2.1 Defining Input and Target (x and y)

In [27]:
x_train = train_df["expression"]
y_train = train_df["value"]
x_val = validation_df["expression"]
y_val = validation_df["value"]

### 2.2 Tokenization

Identify all unique characters from expression to build a vocabulary

In [28]:
all_chars = set()
for expression in x_train:
    for char in expression:
        all_chars.add(char)


V = sorted(list(all_chars))
PAD_TOKEN = "[PAD]"
V = [PAD_TOKEN] + V + [" "]
print(len(V))
token_to_index = {token: i for i, token in enumerate(V)}
print(token_to_index)

16
{'[PAD]': 0, '(': 1, ')': 2, '+': 3, '-': 4, '0': 5, '1': 6, '2': 7, '3': 8, '4': 9, '5': 10, '6': 11, '7': 12, '8': 13, '9': 14, ' ': 15}


In [29]:
def encode(x):
    return [token_to_index[token] for token in list(x)]

def decode(x):
    return ''.join([V[index] for index in x if V[index] != PAD_TOKEN])

# Verify that one-hot encoding and decodign is working as intended
print(encode('+'))
print(decode(encode('+')))

[3]
+


### 2.3 Batching and Transformation

In [30]:
def collate_fn(batch):                #we dynamically pad every batch that has to be given to the nn by the length of the maximum element in it
    # 'batch' is a list of (input_sequence, target_value) tuples
    # For us, input_sequence is x_data, and target_value is y_data

    # Separate inputs (expressions) and targets (values)
    expressions = [item[0] for item in batch]
    values = [item[1] for item in batch]

    # Encode expressions
    encoded_expressions = [encode(expr) for expr in expressions]

    # Find the maximum length in the current batch
    batch_max_length = max(len(seq) for seq in encoded_expressions)

    # Pad each sequence in the batch to batch_max_length
    padded_expressions = []
    for seq in encoded_expressions:
        padding_needed = batch_max_length - len(seq)
        padded_seq = seq + [token_to_index[PAD_TOKEN]] * padding_needed
        padded_expressions.append(padded_seq)

    # Convert to PyTorch tensors
    x_batch = torch.tensor(padded_expressions, dtype=torch.long)
    y_batch = torch.tensor(values, dtype=torch.float)

    return x_batch, y_batch

### 2.4 Model Definition

In [31]:
VOCAB_SIZE = len(V)
#EMBEDDING_DIM = 64 # Dimension for token embeddings
NUM_OUTPUTS = 1 # For regression, we predict a single value (the result of the expression)

class ExpressionCNNOneHot(nn.Module):                               #we are defining a new model to enable the dynamic padding of the inputs
    def __init__(self, vocab_size, hidden_channels=64, num_outputs=1):            #num_outputs is for classification against regression problems, 199 for classification, 1 for regression
        super(ExpressionCNNOneHot, self).__init__()                         #all the attributes of the nn.Module class, (we need to see it a little bit)

        self.conv1 = nn.Conv1d(                                         #in this type of model we add 2 convolution, both of size 3, the inputs of the first one have the vocab size as dimension, the outputs have dimension hidden_channels
            in_channels=vocab_size,
            out_channels=hidden_channels,
            kernel_size=3,
            padding=1                                                     #CONVOLUTIONAL padding
        )

        self.conv2 = nn.Conv1d(                                           #the inputs and the outputs of the of the second one have both the size of hidden_channels, we can change this
            in_channels=hidden_channels,
            out_channels=hidden_channels,
            kernel_size=3,
            padding=1
        )
#we can change the structure by make layers of different sizes, not only fixing one hidden dimension
        self.fc = nn.Linear(hidden_channels, num_outputs)                       #output function, if we use classification we have also to use softmax or directly argmax in the training (We have to check it)

    def forward(self, x, mask):
        # x shape: (batch_size, vocab_size, sequence_length)
        # mask shape: (batch_size, sequence_length)

#we should had a normalization of the parameters
        h = F.relu(self.conv1(x))                                                 #first layer with sigma = ReLU
        h = F.relu(self.conv2(h))                                                   #Second layer with sigma = ReLU, h is the output

# Masked mean pooling
# h shape: (batch_size, hidden_channels, sequence_length)

        #after applying the neural network, we need a way to obtain one vector instead of the whole expression, therefore masked mean pooling (mean pooling but we don't count PAD tokens)
        #mask will be a tensor of the same shame as the input sequence, an element in this tensor can be 1, if the corresponding term of the sequence is not PAD, and 0, if it is PAD

        mask = mask.unsqueeze(1)  # (batch_size, 1, sequence_length)                #to match the output, we have to add antoher dimension corresponding to the hidden channels of h, before this mask has shape (batch_size, max_sequence_length), after this it has shape (batch_size, 1, sequence length). The PyTorch's broadcasting rule comes in action in the multiplication letting this added dimension expand to the size of hidden_channels.
        h = h * mask            #we multiply element wise the output h by mask, possible becuase of the unsqueezing. in the multiplication, every PAD token will be multiplied by zero for every hidden channel

        lengths = mask.sum(dim=2).clamp(min=1)  # (batch_size, 1). mask.sum(dim=2) sums all the elements of the last dimension (sequence_length): given that we applied already multiplied by mas this will give us the actual length of the inputs. clamp(min = 1) is for avoiding lengths being zero (if they are only made by PAD tokens)

        pooled = h.sum(dim=2) / lengths  # (batch_size, hidden_channels). it sums all the characters in every elements of (batch_size, hidden_channels) (i. e. every expression in every hidden channel) and divides by the real length (masked mean pooling).

        output = self.fc(pooled).squeeze(1)       #with the pooled output we apply the final linear transfomration, squeezing at the end the hidden_channels part therefore obtaining a vector of size = batch_size in which every number is the prediction of the value of the expression.

        return output

In [32]:
expression_cnn_one_hot_model = ExpressionCNNOneHot(
    vocab_size=VOCAB_SIZE,
    hidden_channels=64,
    num_outputs=1
)

### 2.5 Training

In [33]:
def train(model, x_train_data, y_train_data, optimizer, loss_fn, epochs, device, batch_size, x_val_data=None, y_val_data=None):
    model.to(device)
    # Create DataLoader for training data
    train_dataset = list(zip(x_train_data, y_train_data))             #we don't use train_df because it has more than 2 elements, it wouldn' know which are the inputs and which are the outputs. given that we already splitted the train_df dataset, it is easier to concatenate these two elements
                                                                    #zip paires every i-th element of x_train_data with the i-th element of y_train_data. list creates a list containing as every elements tuples of (input, output)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

    # Create DataLoader for validation data if provided
    val_loader = None
    if x_val_data is not None and y_val_data is not None:
        val_dataset = list(zip(x_val_data, y_val_data))
        val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)        #collate_fn does the padding of every element in a batch

    for epoch in range(epochs):
        model.train()
        losses = []
        for batch_idx, (x_batch, y_batch) in enumerate(train_loader):               #enumerate is useful to enumerate every batch
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            mask = (x_batch != token_to_index[PAD_TOKEN]).float()             #the mask useful for the forward method in the definition of the class.

            # One-hot encode x_batch and permute for conv1d
            x_batch = F.one_hot(x_batch, num_classes=VOCAB_SIZE).float().permute(0, 2, 1)      #the one_hot sends every token in every expression of the x_batch in a vectore inside R^(VOCAB_SIZE), float converts every element into a vector of floats.
                                                                                                #permute(0, 2, 1) is because the convolutional layers usually want an input of shapes (batch_size, VOCAB_SIZE, sequence_length), right now it is (batch_size, sequence_length, VOCAB_SIZE)
            optimizer.zero_grad()
            y_pred = model(x_batch, mask)       #in nn.Modules, whenever you call model() you are actually doing model.forward(), therefore this would actually be model.forward()
            loss = loss_fn(y_pred, y_batch)

            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        train_loss = np.mean(losses)
        print(f'Train Epoch: {epoch + 1}, Train Loss = {train_loss:.4f}')

        if val_loader is not None:
            val_loss = evaluate(model, val_loader, loss_fn, device)
            print(f'Validation Loss: {val_loss:.4f}')

def evaluate(model, loader, loss_fn, device):
    model.to(device)
    model.eval()
    total_loss = 0
    num_samples = 0
    # The 'correct' variable is not used in a regression context, so it will be removed.
    # correct = 0 # No longer relevant for regression metrics

    with torch.no_grad():
        for batch_size, (x_batch, y_batch) in enumerate(loader):
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            mask = (x_batch != token_to_index[PAD_TOKEN]).float()

            # One-hot encode x_batch and permute for conv1d
            x_batch = F.one_hot(x_batch, num_classes=VOCAB_SIZE).float().permute(0, 2, 1)

            y_pred = model(x_batch, mask)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item() * x_batch.size(0)         #we add to the total loss the average loss in the x_batch as many times as the elements in x_batch (remember x_batch = (batch_size, VOCAB_SIZE, sequence_length))
            num_samples += x_batch.size(0)
            # The following line is for classification accuracy and is not relevant for regression.
            # correct += y_pred.eq(y_batch.view_as(y_pred)).sum().item()

    # The 'accuracy' calculation is for classification and will be removed.
    # accuracy = correct/num_samples
    average_loss = total_loss / num_samples
    # Updated print statement for regression, removing accuracy.
    print(f'Validation set: Average loss: {average_loss:.4f}')
    return average_loss

In [34]:
batch_size = 32
learning_rate = 0.001
epochs = 40
optimizer = torch.optim.Adam(expression_cnn_one_hot_model.parameters(), lr = learning_rate)
loss_fn = nn.MSELoss() # Changed from CrossEntropyLoss to MSELoss for regression
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train(expression_cnn_one_hot_model, x_train, y_train, optimizer, loss_fn, epochs, device, batch_size, x_val, y_val)          #it is normal for regression, we should normalize the datas and the rescale them, the important thing is that it is decreasing

Train Epoch: 1, Train Loss = 84.0726
Validation set: Average loss: 58.0252
Validation Loss: 58.0252
Train Epoch: 2, Train Loss = 39.9930
Validation set: Average loss: 31.1990
Validation Loss: 31.1990
Train Epoch: 3, Train Loss = 26.9356
Validation set: Average loss: 24.2383
Validation Loss: 24.2383
Train Epoch: 4, Train Loss = 22.3450
Validation set: Average loss: 20.1964
Validation Loss: 20.1964
Train Epoch: 5, Train Loss = 18.8952
Validation set: Average loss: 17.6025
Validation Loss: 17.6025
Train Epoch: 6, Train Loss = 16.5128
Validation set: Average loss: 15.7575
Validation Loss: 15.7575
Train Epoch: 7, Train Loss = 15.2833
Validation set: Average loss: 16.2966
Validation Loss: 16.2966
Train Epoch: 8, Train Loss = 14.3743
Validation set: Average loss: 14.0175
Validation Loss: 14.0175
Train Epoch: 9, Train Loss = 13.7354
Validation set: Average loss: 13.7342
Validation Loss: 13.7342
Train Epoch: 10, Train Loss = 13.2734
Validation set: Average loss: 13.1564
Validation Loss: 13.1564

### 2.6 Training (Normalized Value)
By normalizing the outputs, the model should now train on a scaled version of the target values. This often leads to a more stable training process and potentially better convergence. The loss values reported will be for the normalized predictions. To get the actual predicted values, you would need to denormalize the model's output using (prediction * y_std) + y_mean. Given that the output usually is an integer we should also transform the data back into an integer

In [35]:
y_train_normalized = train_df["normalized_value"]
y_val_normalized = validation_df["normalized_value"]

In [36]:
expression_cnn_one_hot_model_normal = ExpressionCNNOneHot(
    vocab_size=VOCAB_SIZE,
    hidden_channels=64,
    num_outputs=1
)

In [ ]:
# Reset optimizer as model has been re-instantiated
optimizer_normal = torch.optim.Adam(expression_cnn_one_hot_model_normal.parameters(), lr=learning_rate)

train(expression_cnn_one_hot_model_normal, x_train, y_train_normalized, optimizer_normal, loss_fn, epochs, device, batch_size, x_val, y_val_normalized)

Train Epoch: 1, Train Loss = 0.4947
Validation set: Average loss: 0.2605
Validation Loss: 0.2605
Train Epoch: 2, Train Loss = 0.2074
Validation set: Average loss: 0.2103
Validation Loss: 0.2103
Train Epoch: 3, Train Loss = 0.1625
Validation set: Average loss: 0.1541
Validation Loss: 0.1541
Train Epoch: 4, Train Loss = 0.1386
Validation set: Average loss: 0.1315
Validation Loss: 0.1315
Train Epoch: 5, Train Loss = 0.1244
Validation set: Average loss: 0.1231
Validation Loss: 0.1231
Train Epoch: 6, Train Loss = 0.1142
Validation set: Average loss: 0.1133
Validation Loss: 0.1133
Train Epoch: 7, Train Loss = 0.1075
Validation set: Average loss: 0.1272
Validation Loss: 0.1272
Train Epoch: 8, Train Loss = 0.1024
Validation set: Average loss: 0.1129
Validation Loss: 0.1129
Train Epoch: 9, Train Loss = 0.0993
Validation set: Average loss: 0.0985
Validation Loss: 0.0985
Train Epoch: 10, Train Loss = 0.0954
Validation set: Average loss: 0.0953
Validation Loss: 0.0953
Train Epoch: 11, Train Loss =

### 2.7 Training (Postfix)

In [ ]:
x_train_postfix = train_df["postfix"]
y_train_normalized = train_df["normalized_value"]
x_val_postfix = validation_df["postfix"]
y_val_normalized = validation_df["normalized_value"]

In [ ]:
expression_cnn_one_hot_model_postfix = ExpressionCNNOneHot(
    vocab_size=VOCAB_SIZE,
    hidden_channels=64,
    num_outputs=1
)

In [ ]:
# Reset optimizer as model has been re-instantiated
optimizer_postfix = torch.optim.Adam(expression_cnn_one_hot_model_postfix.parameters(), lr=learning_rate)

train(expression_cnn_one_hot_model_postfix, x_train_postfix, y_train_normalized, optimizer_postfix, loss_fn, epochs, device, batch_size, x_val_postfix, y_val_normalized)